In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab03.ipynb")

# Lab 03

To receive credit for a lab, answer all questions correctly and submit before the deadline.

You must submit this assignment to Gradescope by the on-time deadline. **We strongly encourage you to plan to submit your work to Gradescope several hours before the stated deadline**. This way, you will have ample time to contact staff for submission support. 

---

### Debugging Guide

If you run into any technical issues, we highly recommend checking out the [Debugging Guide](https://mtu.instructure.com/courses/1619791/pages/debugging-guide). In this guide, you can find general questions about Jupyter notebooks / Jupyterhub, Gradescope, and common pandas errors.

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Pandas 

[`pandas`](https://pandas.pydata.org/) is one of the most widely used `Python` libraries in data science. In this lab, you will review commonly used data-wrangling operations/tools in `pandas`. We continue the content from the previous lab and aim to give you familiarity with:

* Aggregating the data (using `.groupby`),
* Filtering the data (using boolean arrays and `groupby.filter`),
* Pivoting (using `.pivot_table`).

In this lab, you are going to use several `pandas` methods. Reminder from the lecture that you may press `shift+tab` on method parameters to see the documentation for that method. For example, if you were using the `drop` method in `pandas`, you could press `shift+tab` to see what `drop` is expecting.

For those who took DATA 1202, `pandas` is very similar to the `datascience` library that you saw in in that course. This [conversion notebook](https://mtu.instructure.com/courses/1619791/files/folder/lab/week3?preview=134448867) may serve as a useful guide!

This lab expects that you have gone through  `pandas` lectures. If you have not, this lab will probably take a very long time.

In [ ]:
import numpy as np
import pandas as pd
%matplotlib inline

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## **REVIEW:** `Groupby` and `Groupby` Shorthand

Let's now turn to use `groupby` from lectures 4 and 5 (Pandas, part III and IV).

### Elections

Let's start by reading in the election dataset from the `pandas` lectures.

In [ ]:
# Run this cell to load data from CSV file; no further action is needed.
elections = pd.read_csv("data/elections.csv")
elections.head(5)

As we saw before, we can `groupby` a specific column, e.g., `"Party"` and can print out the resulting sub-DataFrames. The output below can help you get an understanding of what `groupby` is actually doing.

An example is given below for elections since 1980.

In [ ]:
# Run this cell to print sub-DataFrames of a groupby object; no further action is needed.
for n, g in elections[elections["Year"] >= 1980].groupby("Party"):
    print(f"Name: {n}") # By the way, this is an "f string", a relatively new and great feature of Python
    display(g)

Recall that once we've formed groups, we can aggregate each sub-DataFrame (a.k.a. group) into a single row using an aggregation function. For example, if we use `.agg('mean')` on the groups above, we get back a single `DataFrame` where each group has been replaced by a single row. In each column for that aggregate row, the value that appears is the average of all values in that group.

For columns that are non-numeric, e.g., `"Result"`, the `pandas` version we're using (version 2.0.2) will error because we cannot compute the mean of the `Result` column. To remedy this, we add a `numeric_only=True` argument to our function calls so that we only calculate the `mean` for columns that contain numeric values. Alternatively, we can manually select only the numerical columns before calling the `agg` function so the aggregation is only applied to numerical columns.

In [ ]:
elections_after_1980 = elections[elections["Year"] >= 1980]

elections_after_1980.groupby("Party").agg('mean', numeric_only=True)

# alternatively, we can manually select only the numerical columns before calling `agg`
# elections_after_1980.groupby("Party")[['Year', 'Popular vote', '%']].agg('mean')

Equivalently we can use one of the shorthand aggregation functions, e.g. `.mean()`: 

In [ ]:
elections_after_1980.groupby("Party").mean(numeric_only=True)

Note that the index of the `DataFrame` returned by an `groupby.agg` call is no longer a set of numeric indices from $0$ to $N-1$. Instead, we see that the index for the example above is now the `Party`. If we want to restore our `DataFrame` so that `Party` is a column rather than the index, we can use `reset_index`.

In [ ]:
elections_after_1980.groupby("Party").mean(numeric_only=True).reset_index()

**IMPORTANT NOTE:** Notice that the code above consists of chained method calls. This sort of code is very common in `pandas` programming and in data science in general. Such chained method calls can sometimes go many layers deep, so you might consider adding newlines between lines of code for clarity. For example, we could instead write the code above as:

In [ ]:
# pandas method chaining
(
elections.query("Year >= 1980").groupby("Party") 
                               .mean(numeric_only=True)  ## Computes the mean values by party
                               .reset_index()            ## Resets to a numerical index
)

Note that we have surrounded the entire call by a big set of parentheses so that `Python` doesn't complain about the indentation. An alternative is to use the \ symbol to indicate to `Python` that your code continues on to the next line!

In [ ]:
# pandas method chaining (alternative)
elections[elections["Year"] >= 1980].groupby("Party") \
                               .mean(numeric_only=True) \
                               .reset_index()     

**IMPORTANT NOTE:** You should NEVER solve problems like the one above using loops or list comprehensions. This is slow and also misses the entire point of this part of DATA 2201. 

Before we continue, we'll print out the election dataset again for your convenience. 

In [ ]:
elections.head(5)

<br/><br/>

---

### Question 1a

Using `groupby.agg` or one of the shorthand aggregation methods (`groupby.max`, `groupby.min`, etc.), create a `Series` named `highest_popular_vote_by_party`.

For each political party, the `Series` should contain the **largest number of popular votes** received by that party in any election.

Keep only parties whose highest popular vote is at least **5,000,000 votes**, and sort the resulting `Series` in **decreasing order**.

The index of your `Series` should be `Party`, and the values should come from the `Popular vote` column. Your result should look like this:

<code>
Party
Democratic              81268924
Republican              74216154
Independent             19743821
American Independent     9901118
Reform                   8085294
Name: Popular vote, dtype: int64
</code>
<br/>
    
A list of named `groupby.agg` shorthand methods is available in the [pandas groupby documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#aggregation).

In [ ]:

...


highest_popular_vote_by_party

In [ ]:
grader.check("q1a")

<br><br>

---

### Question 1b

Repeat the idea from Question 1a, but this time create a `DataFrame` named `highest_vote_election_by_party` that contains the **complete election record** corresponding to each party's highest `Popular vote`.

Your result should:

- contain all available election columns,
- contain one row per party,
- include only parties whose highest `Popular vote` is at least **5,000,000**,
- be sorted from highest to lowest `Popular vote`,
- use `Party` as the index.

Do **not** use `reset_index()`.

Be careful: taking the maximum of every column separately will not necessarily preserve values from the same election row. For example, the first 3 rows of your table should be:

| Party       | Year | Candidate    | Popular vote | Result |         % |
| ----------- | ---: | ------------ | -----------: | ------ | --------: |
| Democratic  | 2020 | Joseph Biden |   81,268,924 | win    | 51.311515 |
| Republican  | 2020 | Donald Trump |   74,216,154 | loss   | 46.858542 |
| Independent | 1992 | Ross Perot   |   19,743,821 | loss   | 18.956298 |



*Hint:* One approach is to sort the original table before using `groupby`.

In [ ]:

...


highest_vote_election_by_party

In [ ]:
grader.check("q1b")

<br><br>


### **REVIEW:** `DataFrameGroupBy.filter`

Our `DataFrame` contains a number of parties that have never had a successful presidential run. For example, the 2020 elections included candidates from the Libertarian and Green parties, neither of which have elected a president.

In [ ]:
# Run this cell to print the last four rows; no further action is needed.
elections.tail(4)

Suppose we were conducting an analysis trying to focus our attention on parties that had elected a president. 

The most natural approach is to use `groupby.filter` [(docs)](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.filter.html). This is an incredibly powerful but subtle tool for filtering data.

The code below accomplishes the task at hand. It does this by creating a function that returns `True` if and only if a sub-`DataFrame` (a.k.a. group) contains at least one winner. This function, in turn, uses the `pandas` function `any` [(docs)](https://pandas.pydata.org/docs/reference/api/pandas.Series.any.html).

In [ ]:
# Run this cell to keep only the rows of parties that have 
# elected a president; no further action is needed.
def at_least_one_candidate_in_the_frame_has_won(frame):
    """Returns df with rows only kept for parties that have
    won at least one election
    """
    return (frame["Result"] == 'win').any()

winners_only = (
    elections
        .groupby("Party")
        .filter(at_least_one_candidate_in_the_frame_has_won)
)
winners_only.tail(5)

Alternately, we could have used a `lambda` function instead of explicitly defining a named function using `def`. 

In [ ]:
# Run this cell to keep only the rows of parties that have 
# elected a president; no further action is needed.
winners_only = (
    elections
        .groupby("Party")
        .filter(lambda x : (x["Result"] == "win").any())
)
winners_only.tail(5)

<br><br>

---

### Question 1c

Using `groupby.filter`, create a `DataFrame` named `competitive_party_results_since_2000`.

First, keep only election results from **2000 onward** (including 2000).

Then group the resulting data by `Party`. Keep **all rows for a party** if that party earned at least **5% of the popular vote in ANY election from 2000 onward**.

For example, the first three rows of the table you generate should look like:

| | Year | Candidate | Party | Popular vote | Result | % |
|---:|---:|---|---|---:|---|---:|
| **151** | 2000 | Al Gore | Democratic | 50999897 | loss | 48.491813 |
| **152** | 2000 | George W. Bush | Republican | 50456002 | win | 47.974666 |
| **157** | 2004 | George W. Bush | Republican | 62040610 | win | 50.771824 |

*Hint:* Consider the following questions:

1. How can you first select only rows where `Year >= 2000`?
2. Which column should you use with `groupby`?
3. How can `.any()` determine whether at least one value in a party's `%` column is greater than or equal to 5?

In [ ]:

...


competitive_party_results_since_2000.head()

In [ ]:
grader.check("q1c")

<br><br>

### **REVIEW:** `str`

`pandas` provides special purpose functions for working with specific common data types such as strings and dates, which you will learn about in more detail in upcoming weeks. For example, the code below provides the length of every Candidate's name from our `elections` dataset. 

In [ ]:
elections["Candidate"].str.len()

<br><br>

---

### Question 2

Using `.str.split`, create a new `DataFrame` named `elections_with_name_parts`.

It should contain all columns from `elections` plus two new columns:

- `First Name`: the first word in the candidate's name
- `Last Name`: the last word in the candidate's name

For example:

- `Andrew Jackson` should have `First Name` equal to `Andrew` and `Last Name` equal to `Jackson`.
- `George H. W. Bush` should have `First Name` equal to `George` and `Last Name` equal to `Bush`.

Do not modify the original `elections` DataFrame.

See the [`pandas.Series.str.split` documentation](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html) if needed.

In [ ]:

elections_with_name_parts = elections.copy()

...


elections_with_name_parts

In [ ]:
grader.check("q2")

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Babynames
Remember the `babynames` dataset from Lab02? Let's load it in again and explore the data with our newly covered functions! Like last time, we'll only load in data from Michigan. 

Run the following cell: 

In [ ]:
file_path = 'data/STATE.MI.TXT'
column_labels = ['State', 'Sex', 'Year', 'Name', 'Count']

babynames = pd.read_csv(file_path, names=column_labels)

babynames.head()

The code below creates a table with the frequency of all names from 2022. 

In [ ]:
# Run this cell to create a table with the frequency 
# of all names from 2022; no further action is needed.
babynames_2022 = (
    babynames[babynames['Year'] == 2022]
              .groupby("Name")
              .sum()[["Count"]]
              .reset_index()
)
babynames_2022

<br><br>

---

### Question 3

Using `pd.merge`, combine `elections_with_name_parts` with `babynames_2022` to create a `DataFrame` named `candidate_name_popularity_2022`.

Match each candidate's `First Name` with the `Name` column in `babynames_2022`.

Use `elections_with_name_parts` as the **left table** and perform a **left join** so that every election record remains in the resulting DataFrame, even when the candidate's first name does not appear in the 2022 Michigan baby-name data.

Your resulting table should contain all columns from both input tables.

See the [`pd.merge` documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) if needed.

In [ ]:

candidate_name_popularity_2022 = ...


candidate_name_popularity_2022

In [ ]:
grader.check("q3")

<br><br>
### **REVIEW:** `pandas.pivot_table`

Suppose we want to build a table showing the total number of babies born of each sex in each year. One way is to `groupby` using both columns of interest:

In [ ]:
babynames.groupby(["Year", "Sex"])[["Count"]].agg("sum").head(6)

While this does give us the information we're looking for, a more natural approach is to use pivot tables to represent our data in a more readable format.

In [ ]:
babynames_pivot = babynames.pivot_table(
    index = "Year",     # rows (turned into index)
    columns = "Sex",    # column values
    values = ["Count"], # field(s) to process in each group
    aggfunc = "sum",   # group operation
)
babynames_pivot.head(6)

We can also include multiple values in our pivot tables 

In [ ]:
babynames_pivot = babynames.pivot_table(
    index = "Year",     # rows (turned into index)
    columns = "Sex",    # column values
    values = ["Count", "Name"],
    aggfunc = "max",   # group operation
)
babynames_pivot.head(6)

<br><br>

---

### Question 4

Using `candidate_name_popularity_2022`, create a pivot table named `party_result_name_count_pivot`.

The table should have:

- `Party` as its index,
- `Result` as its columns,
- `Count` as its values,
- the **mean** `Count` for each Party/Result combination.

Replace any missing values in the resulting pivot table with `0`.

The table therefore summarizes the average 2022 Michigan baby-name frequency of presidential candidates' first names, grouped by political party and election result.

You may use the `fill_value=` argument of `pivot_table` to replace missing values.

See the [`pandas.pivot_table` documentation](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html).

In [ ]:

...


party_result_name_count_pivot

In [ ]:
grader.check("q4")

Just for fun: Which historical presidential candidates have names that were the least and most popular in 2022? Note: Here you'll observe a common problem in data science -- one of the least popular names is actually due to the fact that one recent president was so commonly known by his nickname that he appears named as such in the database from which you pulled election results.

In [ ]:
# your optional code here
...

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Congratulations! You have finished Lab 03!

### Submission Instructions

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers. Submit this file to the Lab 03 assignment on Gradescope. 

### Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)